# Same Anwsers on Test Anomaly

In [1]:
import pandas as pd
import pyodbc  
import matplotlib.pyplot as plt
import numpy as np

## Import Data from csv

In [2]:
fca_test = pd.read_csv("../../../decoded_data/FCA/FactTest.csv")
fca_question = pd.read_csv("../../../decoded_data/FCA/FactQuestionFCA.csv")

## Data Inspection

In [3]:
fca_question.head()

,QuestionKey,TestKey,Competence1Key,Competence2Key,Competence3Key,Competence4Key,ItemId,Answer1,Answer2,Answer3,TimeSpent
0,1,1,1,0,0,0,222,0,0,0,11
1,2,1,2,183,0,0,223,3,4,2,3
2,3,1,3,184,0,0,226,4,3,2,13
3,4,1,4,0,0,0,229,2,0,4,100
4,5,1,5,0,0,0,238,0,0,0,2


In [4]:
fca_question.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3035485 entries, 0 to 3035484
Data columns (total 11 columns):
 #   Column          Dtype
---  ------          -----
 0   QuestionKey     int64
 1   TestKey         int64
 2   Competence1Key  int64
 3   Competence2Key  int64
 4   Competence3Key  int64
 5   Competence4Key  int64
 6   ItemId          int64
 7   Answer1         int64
 8   Answer2         int64
 9   Answer3         int64
 10  TimeSpent       int64
dtypes: int64(11)
memory usage: 254.7 MB


## Data Preparation

In [5]:
df = fca_question[["TestKey", "Answer1", "Answer2", "Answer3"]]
df.head(25)

,TestKey,Answer1,Answer2,Answer3
0,1,0,0,0
1,1,3,4,2
2,1,4,3,2
3,1,2,0,4
4,1,0,0,0
5,2,4,2,5
6,2,3,4,2
7,2,1,2,5
8,2,4,5,2
9,2,1,4,3


In [6]:
# Groepeer de data per CandidateId, en behoud ook InstanceId
all_test_answers = df.groupby(['TestKey']).agg({
    'Answer1': list,
    'Answer2': list,
    'Answer3': list
}).reset_index()

# Voeg alle antwoorden samen in een enkele array
all_test_answers['AllAnswers'] = all_test_answers.apply(
    lambda row: np.array(row['Answer1'] + row['Answer2'] + row['Answer3']),
    axis=1
)

# Behoud de kolommen CandidateId, InstanceId en AllAnswers
all_test_answers = all_test_answers[['TestKey', 'AllAnswers']]
all_test_answers

,TestKey,AllAnswers
0,1,"[0, 3, 4, 2, 0, 0, 4, 3, 0, 0, 0, 2, 2, 4, 0]"
1,2,"[4, 3, 1, 4, 1, 4, 5, 1, 1, 3, 1, 2, 5, 2, 2, ..."
2,3,"[2, 3, 1, 5, 4, 3, 4, 0, 2, 4, 0, 5, 4, 3, 4, ..."
3,4,"[3, 5, 5, 3, 3, 1, 5, 3, 3, 5, 1, 2, 1, 1, 5, ..."
4,5,"[1, 5, 1, 4, 4, 1, 3, 5, 2, 5, 4, 1, 5, 1, 4, ..."
...,...,...
160059,160060,"[4, 1, 4, 4, 3, 1, 5, 4, 3, 5, 3, 3, 4, 3, 4, ..."
160060,160061,"[3, 2, 1, 3, 1, 1, 5, 4, 5, 5, 4, 0, 3, 3, 4, ..."
160061,160062,"[5, 0, 5, 3, 3, 4, 5, 1, 0, 0, 5, 2, 4, 0, 3, ..."
160062,160063,"[1, 0, 5, 4, 0, 4, 1, 3, 0, 3, 4, 4, 0, 0, 3, ..."


## Detection function

In [8]:
def detect_same_answers(FCATestKey, print_result=False):
    answers = all_test_answers[all_test_answers['TestKey'] == FCATestKey].reset_index().AllAnswers[0]

    ans_count = {
        0:0, 
        1:0, 
        2:0,
        3:0,
        4:0,
        5:0
        }
    total_count = 0

    for i in answers: 
        ans_count[i] += 1
        total_count += 1

    max_count = max(ans_count.values())
    most_common_ans = max(ans_count, key=ans_count.get)
    perc = (max_count/total_count) * 100
    if print_result:
        print(f"Candidate {FCATestKey} answered {most_common_ans} on {perc}% of the questions.")
    return perc


In [9]:
detect_same_answers(1, print_result=True)

Candidate 1 answered 0 on 46.666666666666664% of the questions.


46.666666666666664

In [10]:
detect_same_answers(2, print_result=True)

Candidate 2 answered 2 on 28.07017543859649% of the questions.


28.07017543859649

In [11]:
detect_same_answers(3, print_result=True)

Candidate 3 answered 4 on 27.27272727272727% of the questions.


27.27272727272727

## Exporting Data with Anomaly Check

In [12]:
all_test_answers['SameAnswerPercentage'] = all_test_answers['TestKey'].apply(detect_same_answers)
all_test_answers

,TestKey,AllAnswers,SameAnswerPercentage
0,1,"[0, 3, 4, 2, 0, 0, 4, 3, 0, 0, 0, 2, 2, 4, 0]",46.666667
1,2,"[4, 3, 1, 4, 1, 4, 5, 1, 1, 3, 1, 2, 5, 2, 2, ...",28.070175
2,3,"[2, 3, 1, 5, 4, 3, 4, 0, 2, 4, 0, 5, 4, 3, 4, ...",27.272727
3,4,"[3, 5, 5, 3, 3, 1, 5, 3, 3, 5, 1, 2, 1, 1, 5, ...",31.578947
4,5,"[1, 5, 1, 4, 4, 1, 3, 5, 2, 5, 4, 1, 5, 1, 4, ...",33.333333
...,...,...,...
160059,160060,"[4, 1, 4, 4, 3, 1, 5, 4, 3, 5, 3, 3, 4, 3, 4, ...",30.769231
160060,160061,"[3, 2, 1, 3, 1, 1, 5, 4, 5, 5, 4, 0, 3, 3, 4, ...",30.952381
160061,160062,"[5, 0, 5, 3, 3, 4, 5, 1, 0, 0, 5, 2, 4, 0, 3, ...",25.000000
160062,160063,"[1, 0, 5, 4, 0, 4, 1, 3, 0, 3, 4, 4, 0, 0, 3, ...",31.250000


In [13]:
all_test_answers.drop(columns=["AllAnswers"], inplace=True)
all_test_answers

,TestKey,SameAnswerPercentage
0,1,46.666667
1,2,28.070175
2,3,27.272727
3,4,31.578947
4,5,33.333333
...,...,...
160059,160060,30.769231
160060,160061,30.952381
160061,160062,25.000000
160062,160063,31.250000


In [14]:
all_test_answers['is_anomaly'] = all_test_answers.SameAnswerPercentage >= 50
all_test_answers.is_anomaly = all_test_answers.is_anomaly.astype(int)
all_test_answers.head()

,TestKey,SameAnswerPercentage,is_anomaly
0,1,46.666667,0
1,2,28.070175,0
2,3,27.272727,0
3,4,31.578947,0
4,5,33.333333,0


In [15]:
all_test_answers.to_csv("../csv/same_answers_test_checked.csv")